# Safe Signs

CPU only. Upload the project folder to Google Drive as `safe-signs`, then run the cells in order. Results are written to the project folder on Drive, so a disconnect loses nothing.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/safe-signs')
print(os.getcwd())
!pip install -q -r requirements.txt

## 2. Download GTSRB (once)

These are the same files torchvision uses: training images, test images and test labels.

In [ ]:
import os
BASE = 'https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/'
os.makedirs('data/gtsrb', exist_ok=True)
for f in ['GTSRB-Training_fixed.zip', 'GTSRB_Final_Test_Images.zip', 'GTSRB_Final_Test_GT.zip']:
    if not os.path.exists(f'data/gtsrb/{f}'):
        !wget -q -O data/gtsrb/{f} {BASE}{f}
    !unzip -q -o data/gtsrb/{f} -d data/gtsrb
!find data/gtsrb -name '*.ppm' | wc -l
!find data/gtsrb -name 'GT-final_test.csv'

## 3. Check the code (synthetic signs, a few minutes)

In [ ]:
!python -m pytest -q
!python scripts/run.py --smoke | tail -3

## 4. Real map contexts from OpenStreetMap (about 15-40 min)

Downloads German traffic signs with the road network around them for Berlin, Potsdam, München, Hamburg and one rural district (Landkreis Märkisch-Oderland). Junctions now come from shared road nodes. This is a fresh download (files named osm_v2_*), so the old Berlin/Potsdam files are not reused. If Overpass refuses an area it is skipped; run the cell again later to add it. Downloaded areas are cached.

In [ ]:
!python scripts/build_osm_contexts.py

## 5. Run the study

`--reuse-weights` reuses `results/main/cnn_weights.npz`, `cnn_adv_weights.npz` and any `cnn_weights_s1/_s2.npz`; the first lines of the output say whether each file was found. This run adds rounds three and four: soft rules, two extra training seeds (about 25-30 min each, only the first time), the envelope check, the rule-aware adaptive attack, map spoofing, the attack strength curve, the baselines table and the failure analysis.

To save about an hour, skip the extra seeds by adding `--no-seeds`. Everything is saved to Drive as it goes, so after a disconnect just run the cell again.

In [ ]:
!python scripts/run.py --reuse-weights
# faster: !python scripts/run.py --reuse-weights --no-seeds

## 5. Results

In [ ]:
from IPython.display import Markdown, Image, display
import glob
display(Markdown(open('results/main/summary.md').read()))
for p in sorted(glob.glob('results/main/figures/*.png')):
    display(Image(p, width=750))